# 6.1 · K-Means 聚类 / K-Means Clustering

> **课程定位 / Where this fits**
> Part 6 第 1 课, 也是无监督学习的"Hello World"。**没有标签**, 目标是把相似样本归到一起。K-Means 是最经典的聚类: 给定簇数 K, 反复"分配→更新质心", 让簇内平方距离最小。它直观、快, 但有一堆要注意的坑(选 K、初始化、形状假设)。
> The Hello World of unsupervised learning: group similar points with no labels. K-Means alternates assign→update to minimise within-cluster variance.

> 💡 **面试相关 / Interview-relevant**
> - "K-Means 目标函数 / 为什么会收敛" ★★★★★
> - "K-Means++ 解决什么" ★★★★★（初始化敏感）
> - "怎么选 K(elbow / silhouette)" ★★★★★
> - "K-Means 的假设和失效场景" ★★★★★（球形等方差簇）
> - "K-Means 为什么要缩放" ★★★★
> - "K-Means vs GMM" ★★★★（硬分配 vs 软分配）

---

## 学习目标 / Learning Objectives
1. K-Means 目标(惯性/inertia)与 Lloyd 算法。
2. 从零实现 + 对照 sklearn。
3. **K-Means++** 初始化为何重要。
4. 选 K: **肘部法 + 轮廓系数(silhouette)**。
5. 失效场景(非球形/不同密度)与缩放。

## 目录 / TOC
1. [目标函数与 Lloyd 算法 ⭐](#1)
2. [🛍️ 数据: Mall Customers](#2)
3. [从零实现 + 对照 sklearn ⭐](#3)
4. [K-Means++ 与初始化 ⭐](#4)
5. [选 K: 肘部 + 轮廓 ⭐](#5)
6. [失效场景与缩放 ⭐](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 目标函数与 Lloyd 算法 ⭐ / Objective & Lloyd's Algorithm

把 $n$ 个点分到 $K$ 个簇, 每簇一个质心 $\boldsymbol\mu_k$。最小化**簇内平方和(inertia / WCSS)**:
$$J = \sum_{k=1}^{K}\sum_{\mathbf{x}_i\in C_k} \|\mathbf{x}_i - \boldsymbol\mu_k\|^2$$

直接最优是 NP 难。**Lloyd 算法**交替两步(坐标下降):
1. **分配**: 每个点归到最近质心的簇。
2. **更新**: 每个质心 = 该簇所有点的均值(均值最小化平方距离, 0.x 节学过)。

**为何收敛**: 两步都不增大 $J$, 且 $J\ge0$ 有下界 → 单调收敛。但只保证**局部最优**(依赖初始化, 故需 K-Means++ + 多次重启)。


<a id="2"></a>
## 2. 数据: Mall Customers / The Mall Customers Dataset

经典聚类教学集——商场会员卡数据。每行一位顾客, 特征: 年龄、年收入(k$)、消费分(1–100, 商场依消费行为打的分)。任务: 无标签地发现**顾客分群**(如"高收入高消费""低收入谨慎"), 做精准营销。这里内联合成一个忠于原数据结构(5 个自然客群)的版本并介绍它。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=3, suppress=True)

def make_mall(seed=0):
    rng = np.random.default_rng(seed)
    # 5 个真实客群: (年收入 k$, 消费分) 中心 + 年龄倾向
    groups = [((55, 50), 120, (25,60)),   # 中等收入中等消费(主流)
              ((25, 80), 35,  (18,35)),   # 低收入高消费(年轻冲动)
              ((85, 82), 40,  (28,42)),   # 高收入高消费(目标客户)
              ((85, 18), 38,  (35,60)),   # 高收入低消费(谨慎富人)
              ((26, 18), 35,  (40,68))]   # 低收入低消费
    rows = []
    for (inc, spd), n, (amin,amax) in groups:
        income = rng.normal(inc, 8, n).clip(15, 140)
        spend = rng.normal(spd, 9, n).clip(1, 99)
        age = rng.integers(amin, amax, n)
        rows.append(np.c_[age, income, spend])
    X = np.vstack(rows); rng.shuffle(X)
    return pd.DataFrame(X, columns=["age","income_k","spending"])

mall = make_mall()
print(f"Mall Customers: {mall.shape}")
print(mall.describe().round(1).to_string())

fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(mall["income_k"], mall["spending"], s=20, alpha=0.6)
ax.set_xlabel("年收入 (k$)"); ax.set_ylabel("消费分 (1-100)")
ax.set_title("Mall Customers: 肉眼可见若干自然客群(待聚类发现)")
plt.tight_layout(); plt.show()


<a id="3"></a>
## 3. 从零实现 + 对照 sklearn ⭐ / From Scratch

聚类前**缩放**(下节详述)。先在 income/spending 两维上做(便于可视化)。


In [ ]:
from sklearn.preprocessing import StandardScaler
X2 = mall[["income_k","spending"]].values
Xs = StandardScaler().fit_transform(X2)

def kmeans_scratch(X, K, n_iter=100, seed=0):
    rng = np.random.default_rng(seed)
    mu = X[rng.choice(len(X), K, replace=False)]    # 随机初始质心
    for _ in range(n_iter):
        d = ((X[:,None,:] - mu[None,:,:])**2).sum(-1)  # n×K 距离²
        labels = d.argmin(1)                            # 分配
        new_mu = np.array([X[labels==k].mean(0) if (labels==k).any() else mu[k]
                           for k in range(K)])          # 更新=均值
        if np.allclose(new_mu, mu): break
        mu = new_mu
    inertia = ((X - mu[labels])**2).sum()
    return labels, mu, inertia

labels, mu, inertia = kmeans_scratch(Xs, 5)
print(f"从零 K-Means inertia: {inertia:.1f}")

from sklearn.cluster import KMeans
km = KMeans(n_clusters=5, n_init=10, random_state=0).fit(Xs)
print(f"sklearn K-Means inertia: {km.inertia_:.1f}")

fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(Xs[:,0], Xs[:,1], c=labels, cmap="tab10", s=20, alpha=0.7)
ax.scatter(mu[:,0], mu[:,1], c="black", marker="X", s=200, label="质心")
ax.set_xlabel("income (std)"); ax.set_ylabel("spending (std)"); ax.legend()
ax.set_title("从零 K-Means (K=5): 发现 5 个客群")
plt.tight_layout(); plt.show()


<a id="4"></a>
## 4. K-Means++ 与初始化 ⭐ / Initialization

随机初始质心可能让 Lloyd 收敛到**糟糕的局部最优**。**K-Means++** 智能播种: 第一个质心随机, 之后每个新质心以正比于"到已有质心距离²"的概率选——倾向**分散开**。这大幅降低坏局部最优概率, 是 sklearn 默认(`init='k-means++'`)。另外 `n_init` 多次重启取最优 inertia。


In [ ]:
# 对比随机初始化 vs k-means++ 的稳定性 / random vs k-means++
rand_inertias = [KMeans(5, init="random", n_init=1, random_state=s).fit(Xs).inertia_ for s in range(30)]
pp_inertias   = [KMeans(5, init="k-means++", n_init=1, random_state=s).fit(Xs).inertia_ for s in range(30)]
fig, ax = plt.subplots(figsize=(7,4))
ax.hist(rand_inertias, bins=15, alpha=0.6, label="random init")
ax.hist(pp_inertias, bins=15, alpha=0.6, label="k-means++")
ax.set_xlabel("inertia (越低越好)"); ax.set_ylabel("频次"); ax.legend()
ax.set_title("K-Means++ 初始化: inertia 更低更稳定(更少陷坏局部最优)")
plt.tight_layout(); plt.show()
print(f"random init  inertia: 均值 {np.mean(rand_inertias):.1f}, 最差 {max(rand_inertias):.1f}")
print(f"k-means++    inertia: 均值 {np.mean(pp_inertias):.1f}, 最差 {max(pp_inertias):.1f}")


<a id="5"></a>
## 5. 选 K: 肘部 + 轮廓 ⭐ / Choosing K

无标签时怎么定 K? 两个常用工具:
- **肘部法(elbow)**: 画 inertia vs K。inertia 随 K 单调降, 但在"真实 K"后下降变缓——**拐点(肘部)**就是好的 K。
- **轮廓系数(silhouette)**: 每个点 $s = \frac{b-a}{\max(a,b)}$, $a$=到同簇点平均距离, $b$=到最近邻簇平均距离。范围 $[-1,1]$, **越高越好**。取平均轮廓最大的 K。比肘部更客观。


In [ ]:
from sklearn.metrics import silhouette_score
Ks = range(2, 11)
inertias = [KMeans(k, n_init=10, random_state=0).fit(Xs).inertia_ for k in Ks]
sils = [silhouette_score(Xs, KMeans(k, n_init=10, random_state=0).fit_predict(Xs)) for k in Ks]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(list(Ks), inertias, "o-"); axes[0].axvline(5, color="r", ls="--", label="肘部≈5")
axes[0].set_xlabel("K"); axes[0].set_ylabel("inertia"); axes[0].legend(); axes[0].set_title("肘部法")
axes[1].plot(list(Ks), sils, "s-");
best_k = list(Ks)[int(np.argmax(sils))]
axes[1].axvline(best_k, color="r", ls="--", label=f"最佳 K={best_k}")
axes[1].set_xlabel("K"); axes[1].set_ylabel("平均轮廓系数"); axes[1].legend(); axes[1].set_title("轮廓系数")
plt.tight_layout(); plt.show()
print(f"肘部≈5(我们造数据时就是5群); 轮廓系数最佳 K={best_k}")


<a id="6"></a>
## 6. 失效场景与缩放 ⭐ / When K-Means Fails

K-Means 的**隐含假设**: 簇是**球形、大小相近、密度相近**(因为只用到欧氏距离 + 均值)。违反时会失败:
- **非球形**(月牙/环): K-Means 切不出弯曲簇 → 用 DBSCAN(6.4)/谱聚类(6.7)。
- **不同密度/大小**: 大簇会"吞"小簇。
- **不缩放**: 大量纲特征主导距离(同 KNN/SVM 5.3/5.5)→ 聚类前必须缩放。


In [ ]:
from sklearn.datasets import make_moons
Xm, _ = make_moons(300, noise=0.06, random_state=0)
km_moon = KMeans(2, n_init=10, random_state=0).fit_predict(Xm)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].scatter(Xm[:,0], Xm[:,1], c=km_moon, cmap="coolwarm", s=15)
axes[0].set_title("K-Means 在月牙形上失败(只能切直线/球形)")

# 缩放的重要性 / scaling
X_age = mall[["age","income_k"]].values   # age(18-68) vs income(15-140) 量纲差
lab_unscaled = KMeans(5, n_init=10, random_state=0).fit_predict(X_age)
axes[1].scatter(X_age[:,0], X_age[:,1], c=lab_unscaled, cmap="tab10", s=15)
axes[1].set_xlabel("age"); axes[1].set_ylabel("income_k")
axes[1].set_title("不缩放: 簇沿大量纲(income)切分, age 几乎被忽略")
plt.tight_layout(); plt.show()
print("月牙→需 DBSCAN/谱聚类; 量纲不一→聚类前必须标准化")


<a id="7"></a>
## 7. 小结 / Summary

```
K-Means: 最小化簇内平方和(inertia); Lloyd 交替 分配(最近质心)→更新(簇均值)
收敛: 两步都不增 J 且有下界 → 单调收敛到局部最优(依赖初始化)
K-Means++: 按距离²概率分散播种, 降低坏局部最优; n_init 多次重启
选 K: 肘部法(inertia 拐点) + 轮廓系数(越高越好, 更客观)
假设: 球形/等大小/等密度簇 + 欧氏距离 → 非球形(月牙)失败, 必须缩放
```

### 💡 面试速查
1. **目标=inertia(簇内平方和)**; Lloyd 分配+更新, 收敛到局部最优
2. **K-Means++** 分散初始质心, 避免坏局部最优(默认)
3. **选 K**: elbow(拐点) + silhouette(最大); 业务也可定 K
4. **假设球形等密度簇** → 月牙/环形失败 → DBSCAN/谱聚类
5. **必须缩放**(欧氏距离对量纲敏感); 硬分配(vs GMM 软分配)

### 下一节
**6.2 Mini-batch K-Means**——数据上百万时全量 Lloyd 太慢, 用小批近似质心更新, 速度快几十倍, 精度略降。
